In [1]:
# Imports and configuration

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

RANDOM_SEED = 42

In [101]:
# Load Information

transactions = pd.read_excel(r"datasets\synthetic_transactions.xlsx")

In [102]:
# Confirm correct loading
print(transactions.shape)

(464049, 13)


In [63]:
transactions.head()

,tx_id,customer_id,product_id,product_type,tx_timestamp,tx_amount,tx_type,tx_channel,tx_description,municipality,risk_rating,is_suspicious,pattern_tag
0,1,1,1,Cuenta de ahorros,2024-05-07 17:37:37,"5,000.00",debit,POS,Consignación,Medellín,low,0,normal
1,2,1,1,Cuenta de ahorros,2024-10-15 18:56:49,"5,000.00",debit,PSE,Transferencia,Medellín,low,0,normal
2,3,1,1,Cuenta de ahorros,2024-06-06 14:32:03,"5,000.00",debit,Web,Pago nómina,Medellín,low,0,normal
3,4,1,1,Cuenta de ahorros,2024-01-03 11:04:37,"5,000.00",credit,ATM,Compra comercio,Medellín,low,0,normal
4,5,1,1,Cuenta de ahorros,2024-04-14 11:46:51,"5,000.00",debit,Web,Compra comercio,Medellín,low,0,normal


In [64]:
transactions.columns.tolist()

['tx_id',
 'customer_id',
 'product_id',
 'product_type',
 'tx_timestamp',
 'tx_amount',
 'tx_type',
 'tx_channel',
 'tx_description',
 'municipality',
 'risk_rating',
 'is_suspicious',
 'pattern_tag']

In [65]:
# Verify data types

transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 464049 entries, 0 to 464048
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   tx_id           464049 non-null  int64         
 1   customer_id     464049 non-null  int64         
 2   product_id      464049 non-null  int64         
 3   product_type    464049 non-null  object        
 4   tx_timestamp    464049 non-null  datetime64[ns]
 5   tx_amount       464049 non-null  float64       
 6   tx_type         464049 non-null  object        
 7   tx_channel      464049 non-null  object        
 8   tx_description  464049 non-null  object        
 9   municipality    464049 non-null  object        
 10  risk_rating     464049 non-null  object        
 11  is_suspicious   464049 non-null  int64         
 12  pattern_tag     464049 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(7)
memory usage: 46.0+ MB


In [11]:
# Convert correct data types

In [66]:
# Verify duplicates

transactions.duplicated().sum()

np.int64(0)

In [68]:
(
    transactions
    .isnull()
    .mean()
    .sort_values(ascending=False)
    * 100
)

tx_id            0.00
customer_id      0.00
product_id       0.00
product_type     0.00
tx_timestamp     0.00
tx_amount        0.00
tx_type          0.00
tx_channel       0.00
tx_description   0.00
municipality     0.00
risk_rating      0.00
is_suspicious    0.00
pattern_tag      0.00
dtype: float64

In [71]:
transactions["tx_type"].value_counts()

tx_type
debit     285286
credit    178763
Name: count, dtype: int64

### Exploratory Aggregations

In [72]:
# Transactions by product

(
    transactions
    .groupby("product_type")
    .agg(
        transactions=("tx_id", "count"),
        total_amount=("tx_amount", "sum"),
        avg_amount=("tx_amount", "mean")
    )
    .sort_values("total_amount", ascending=False)
)

,transactions,total_amount,avg_amount
product_type,,,
Cuenta de ahorros,376495,"54,943,860,100.61","145,935.17"
Crédito libre inversión,47959,"7,197,459,975.86","150,075.27"
Crédito libranza,30048,"4,515,088,107.63","150,262.52"
CDT,9547,"1,420,472,674.09","148,787.33"


In [73]:
# Transactions by type

(
    transactions
    .groupby("tx_type")
    .agg(
        transactions=("tx_id", "count"),
        total_amount=("tx_amount", "sum"),
        avg_amount=("tx_amount", "mean")
    )
)

,transactions,total_amount,avg_amount
tx_type,,,
credit,178763,"26,348,163,679.07","147,391.59"
debit,285286,"41,728,717,179.12","146,269.77"


In [74]:
# Transactions by channel

(
    transactions
    .groupby("tx_channel")
    .agg(
        transactions=("tx_id", "count"),
        total_amount=("tx_amount", "sum"),
        avg_amount=("tx_amount", "mean")
    )
    .sort_values("transactions", ascending=False)
)

,transactions,total_amount,avg_amount
tx_channel,,,
Mobile App,162571,"23,862,583,039.70","146,782.53"
Web,92881,"13,491,341,196.00","145,254.05"
PSE,69824,"10,101,576,105.00","144,671.98"
ATM,69482,"10,213,414,948.89","146,993.68"
POS,46097,"6,930,323,287.86","150,342.18"
Branch,23194,"3,477,642,280.74","149,937.15"


In [75]:
# product + tx_type

(
    transactions
    .groupby(
        ["product_type", "tx_type"]
    )
    .agg(
        transactions=("tx_id", "count"),
        total_amount=("tx_amount", "sum"),
        avg_amount=("tx_amount", "mean")
    )
)

transactions      total_amount  avg_amount
product_type            tx_type                                            
CDT                     credit           7627  1,088,433,318.15  142,707.92
                        debit            1920    332,039,355.95  172,937.16
Crédito libranza        credit           9113  1,446,131,576.63  158,688.86
                        debit           20935  3,068,956,531.00  146,594.53
Crédito libre inversión credit          12117  1,783,200,152.19  147,165.15
                        debit           35842  5,414,259,823.66  151,059.09
Cuenta de ahorros       credit         149906 22,030,398,632.10  146,961.42
                        debit          226589 32,913,461,468.51  145,256.22

### Customer level aggregation

In [76]:
customer_features = (
    transactions
    .groupby("customer_id")
    .agg(
        total_transactions=("tx_id", "count"),
        total_amount=("tx_amount", "sum"),
        avg_amount=("tx_amount", "mean"),
        max_amount=("tx_amount", "max"),
        debit_transactions=(
            "tx_type",
            lambda x: (x == "debit").sum()
        ),
        credit_transactions=(
            "tx_type",
            lambda x: (x == "credit").sum()
        ),
        unique_products=("product_type", "nunique"),
        unique_channels=("tx_channel", "nunique")
    )
    .reset_index()
)

In [77]:
customer_features.shape

(4908, 9)

In [78]:
# Distribution analysis

customer_features["total_amount"].describe()

count        4,908.00
mean    13,870,595.12
std     12,002,992.16
min         10,000.00
25%        623,263.18
50%     10,090,648.10
75%     19,590,475.25
max     87,578,619.95
Name: total_amount, dtype: float64

In [79]:
customer_features["total_transactions"].describe()

count   4,908.00
mean       94.55
std        28.81
min         2.00
25%        77.00
50%        91.00
75%       108.00
max       250.00
Name: total_transactions, dtype: float64

In [80]:
# Transactions by product

px.bar(
    (
        transactions.groupby("product_type")
        ["tx_id"]
        .count()
        .reset_index()
    ),
    x="product_type",
    y="tx_id",
    title="Transactions by Product"
)

In [81]:
# Transactions by type

px.bar(
    (
        transactions.groupby("tx_type")
        ["tx_id"]
        .count()
        .reset_index()
    ),
    x="tx_type",
    y="tx_id",
    title="Transactions by Type"
)

In [82]:
# Amount distribution

px.histogram(
    transactions.sample(50000),
    x="tx_amount",
    nbins=100,
    title="Transaction Amount Distribution"
)

## Grouped Datasets

### Customer aggregation

In [83]:
customer_agg = (
    transactions
    .groupby(
        [
            "customer_id",
            "product_type",
            "tx_type"
        ]
    )
    .agg(
        transaction_count=("tx_id", "count"),
        transaction_mean_amount=("tx_amount", "mean"),
        transaction_sum_amount=("tx_amount", "sum")
    )
    .reset_index()
)

In [84]:
customer_agg.head()

,customer_id,product_type,tx_type,transaction_count,transaction_mean_amount,transaction_sum_amount
0,1,Cuenta de ahorros,credit,26,"5,000.00","130,000.00"
1,1,Cuenta de ahorros,debit,34,"566,181.94","19,250,185.93"
2,2,Cuenta de ahorros,credit,25,"737,456.45","18,436,411.25"
3,2,Cuenta de ahorros,debit,41,"232,148.27","9,518,079.21"
4,3,Crédito libre inversión,credit,11,"5,000.00","55,000.00"


In [103]:
customer_agg.to_parquet(
    "grouped_tx/customer_tx_aggregations.parquet",
    index=False
)

### Product aggregation

In [86]:
product_agg = (
    transactions
    .groupby(
        [
            "product_type",
            "tx_type"
        ]
    )
    .agg(
        unique_customers=("customer_id", "nunique"),
        transaction_count=("tx_id", "count"),
        transaction_mean_amount=("tx_amount", "mean"),
        transaction_sum_amount=("tx_amount", "sum")
    )
    .reset_index()
)

In [87]:
product_agg.head()

,product_type,tx_type,unique_customers,transaction_count,transaction_mean_amount,transaction_sum_amount
0,CDT,credit,1194,7627,"142,707.92","1,088,433,318.15"
1,CDT,debit,946,1920,"172,937.16","332,039,355.95"
2,Crédito libranza,credit,1496,9113,"158,688.86","1,446,131,576.63"
3,Crédito libranza,debit,1499,20935,"146,594.53","3,068,956,531.00"
4,Crédito libre inversión,credit,1725,12117,"147,165.15","1,783,200,152.19"


In [104]:
product_agg.to_parquet(
    "grouped_tx/product_tx_aggregations.parquet",
    index=False
)

### Channel aggregation

In [89]:
channel_agg = (
    transactions
    .groupby(
        [
            "tx_channel",
            "tx_type"
        ]
    )
    .agg(
        unique_customers=("customer_id", "nunique"),
        transaction_count=("tx_id", "count"),
        transaction_mean_amount=("tx_amount", "mean"),
        transaction_sum_amount=("tx_amount", "sum")
    )
    .reset_index()
)

In [90]:
channel_agg.head()

,tx_channel,tx_type,unique_customers,transaction_count,transaction_mean_amount,transaction_sum_amount
0,ATM,credit,4833,27071,"147,401.15","3,990,296,621.25"
1,ATM,debit,4878,42411,"146,733.59","6,223,118,327.64"
2,Branch,credit,4022,8948,"159,178.77","1,424,331,623.55"
3,Branch,debit,4536,14246,"144,132.43","2,053,310,657.19"
4,Mobile App,credit,4895,62258,"142,923.67","8,898,141,792.31"


In [105]:
channel_agg.to_parquet(
    "grouped_tx/channel_tx_aggregations.parquet",
    index=False
)

### Jurisdiction / municipality aggregation

In [92]:
jurisdiction_agg = (
    transactions
    .groupby(
        [
            "municipality",
            "tx_type"
        ]
    )
    .agg(
        unique_customers=("customer_id", "nunique"),
        transaction_count=("tx_id", "count"),
        transaction_mean_amount=("tx_amount", "mean"),
        transaction_sum_amount=("tx_amount", "sum")
    )
    .reset_index()
)

In [57]:
jurisdiction_agg.head()

,municipality,tx_type,unique_customers,transaction_count,transaction_mean_amount,transaction_sum_amount
0,Armenia,credit,266,9782,"143,447.33","1,403,201,763.47"
1,Armenia,debit,266,15730,"156,642.10","2,463,980,253.53"
2,Barranquilla,credit,274,9811,"148,327.85","1,455,244,564.02"
3,Barranquilla,debit,274,15686,"140,663.80","2,206,452,310.16"
4,Bogotá,credit,228,8099,"135,755.79","1,099,486,127.39"


In [106]:
jurisdiction_agg.to_parquet(
    "grouped_tx/jurisdiction_tx_aggregations.parquet",
    index=False
)

In [94]:
print(customer_agg.shape)
print(product_agg.shape)
print(channel_agg.shape)
print(jurisdiction_agg.shape)

(18090, 6)
(8, 6)
(12, 6)
(40, 6)
